# Discrete choice modelling

## The conditional logit with ModeCanada dataset

In [1]:
from choice_learn.datasets import load_modecanada
from choice_learn.data import ChoiceDataset
from choice_learn.models import ConditionalLogit
import numpy as np
import tensorflow as tf

c:\Users\danil\anaconda3\envs\vd200\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [2]:
#importing dataset
transport_df = load_modecanada(as_frame=True)
print(f"Dataset shape: {transport_df.shape}")
display(transport_df.head(8))

Dataset shape: (15520, 11)


,case,alt,choice,dist,cost,ivt,ovt,freq,income,urban,noalt
0,1,train,0,83,28.25,50,66,4,45.0,0,2
1,1,car,1,83,15.77,61,0,0,45.0,0,2
2,2,train,0,83,28.25,50,66,4,25.0,0,2
3,2,car,1,83,15.77,61,0,0,25.0,0,2
4,3,train,0,83,28.25,50,66,4,70.0,0,2
5,3,car,1,83,15.77,61,0,0,70.0,0,2
6,4,train,0,83,28.25,50,66,4,70.0,0,2
7,4,car,1,83,15.77,61,0,0,70.0,0,2


### Exploring case 1

In [3]:
transport_df.loc[transport_df["case"] == 1,:]

,case,alt,choice,dist,cost,ivt,ovt,freq,income,urban,noalt
0,1,train,0,83,28.25,50,66,4,45.0,0,2
1,1,car,1,83,15.77,61,0,0,45.0,0,2


For case 1 there were two alternative and the car was choen over the train

### Coverting to ChoiceDataset

In [4]:
canada_dataset = ChoiceDataset.from_single_long_df(
df=transport_df,
items_id_column="alt", # identifies each alternative
choices_id_column="case", # identifies each choice situation
choices_column="choice", # indicates which was chosen
shared_features_columns=["income"], # traveler characteristics
items_features_columns=["cost", "freq", "ovt", "ivt"], # alternative attributes
choice_format="one_zero"
)
print(canada_dataset.summary())


%=====================================================================%
%%% Summary of the dataset:
%=====================================================================%
Number of items: 4
Number of choices: 4324
%=====================================================================%
 Shared Features by Choice:
 1 shared features
 with names: (['income'],)


 Items Features by Choice:
4 items features 
 with names: (['cost', 'freq', 'ovt', 'ivt'],)
%=====================================================================%



## Model specification

𝑈𝑖𝑗 = 𝛽𝑖𝑛𝑡𝑒𝑟𝑗 + 𝛽𝑐𝑜𝑠𝑡⋅ cost𝑗 + 𝛽𝑓𝑟𝑒𝑞⋅ freq𝑗 + 𝛽𝑜𝑣𝑡⋅ ovt𝑗+ 𝛽𝑖𝑣𝑡𝑗⋅ ivt𝑗 + 𝛽𝑖𝑛𝑐𝑜𝑚𝑒𝑗⋅ income

The utility specification includes both generic coefficients (for cost, frequency, and out-of-vehicle time) and alternative-specific coefficients (for the alternative intercepts and in-vehicle time). This allows the model to capture both common preferences across modes and mode-specific sensitivities. In particular, the coefficient on in-vehicle time is alternative-specific, reflecting that travellers may value time differently depending on the transport mode. For example, two hours of in-vehicle time may be perceived as much more burdensome when driving than when flying or taking a train, so imposing a single time coefficient across all alternatives would be overly restrictive. 

In [5]:
#model 
model = ConditionalLogit(optimizer="lbfgs")

#adding parameters
J = 4
#shared features
model.add_shared_coefficient(feature_name="ovt", items_indexes=list(range(J)))
model.add_shared_coefficient(feature_name="cost", items_indexes=list(range(J)))
model.add_shared_coefficient(feature_name="freq", items_indexes=list(range(J)))

#base alternative
base_alt = 0 


#alternative specific features
model.add_coefficients(feature_name="income", items_indexes=[j for j in range(J) if j != base_alt])
model.add_coefficients(feature_name="ivt", items_indexes=[j for j in range(J) if j != base_alt])
model.add_coefficients(feature_name="intercept", items_indexes=[j for j in range(J) if j != base_alt])

Using L-BFGS optimizer, setting up .fit() function


In [6]:
history = model.fit(canada_dataset, get_report=True, verbose=1)

Using L-BFGS optimizer, setting up .fit() function


In [7]:
print("\nModel report:")
print(model.report)


Model report:
    Coefficient Name  Coefficient Estimation  Std. Err    z_value  \
0           beta_ovt               -0.040124  0.002207 -18.181631   
1          beta_cost               -0.007260  0.001733  -4.190248   
2          beta_freq                0.075147  0.003933  19.109131   
3      beta_income_0               -0.064509  0.004914 -13.127548   
4      beta_income_1               -0.025757  0.002804  -9.184973   
5      beta_income_2               -0.038797  0.003324 -11.672670   
6         beta_ivt_0               -0.011648  0.001619  -7.194586   
7         beta_ivt_1               -0.015804  0.000650 -24.331371   
8         beta_ivt_2               -0.006260  0.000549 -11.410406   
9   beta_intercept_0                1.119491  0.271081   4.129736   
10  beta_intercept_1                2.776833  0.176100  15.768484   
11  beta_intercept_2                3.258909  0.243876  13.362976   

          P(.>z)  
0   0.000000e+00  
1   2.786503e-05  
2   0.000000e+00  
3   0.00000

The coefficient on cost is negative, which implies that as the cost of an alternative increases, its utility decreases. This is economically intuitive because higher prices reduce the attractiveness of a transport mode, holding other attributes constant. Therefore, individuals are less likely to choose more expensive alternatives.


The mode corresponding to intercept_2 has the highest baseline utility, meaning that, all else equal (same cost, time, etc.), individuals have the strongest intrinsic preference for this mode relative to the base alternative. Mode 2 is train.

Higher income individuals are more likly to choose air travel as all the coefficents are negative realtive to the baseline (air)

### Own price elasticity 

In [8]:
#extracting beta
beta_cost = model.report.loc[
    model.report["Coefficient Name"] == "beta_cost",
    "Coefficient Estimation"
].values[0]

#predicted probabilites
proba = model.predict_probas(canada_dataset)

#mean price and mean probabilties for cars
car_index = 2  # ['air', 'bus', 'car', 'train']

# Extract cost tensor (N x J x K)
X = canada_dataset.items_features_by_choice[0]


# Mean price of car
p_car_mean = X[:, car_index, 0].mean()

# Mean predicted probability of choosing car
P_car_mean =  tf.reduce_mean(proba[:, car_index]).numpy()

#computing eladticities
elasticity_car = beta_cost * p_car_mean * (1 - P_car_mean)

print("Beta_cost:", beta_cost)
print("Mean price (car):", p_car_mean)
print("Mean probability (car):", P_car_mean)
print("Own-price elasticity (car):", elasticity_car)

Beta_cost: -0.007260071
Mean price (car): 63.76371877890976
Mean probability (car): 0.5117911
Own-price elasticity (car): -0.22600611805950624
